# Abdomen CT — nnUNet v2 Pipeline
### Kaggle Notebook (Birincil) · Google Colab (İkincil)

**Kaggle'da çalıştırma:**
1. Sağ panelden Dataset ekle: `ramazan2020/abdomen-acikveri`
2. `Settings → Accelerator → GPU (P100/T4)` seç
3. Hücreleri sırayla çalıştır — veri indirme gerekmez

**Colab'da çalıştırma:**
1. `Runtime → Change runtime type → GPU`
2. Kaggle Secrets'a `KAGGLE_USERNAME` + `KAGGLE_KEY` ekle
3. Google Drive bağlantısı kur (veri Drive'dan okunur)

---

| Adım | Süre (T. yakl.) |
|------|-----------------|
| DICOM → NIfTI | 20–40 dk |
| Plan + Preprocess | 15–30 dk |
| Eğitim 3d_fullres | 8–12 saat |
| Inference + Değerlendirme | 20–30 dk |

> **Eğitim uzun:** Kaggle 9 saat session sınırı vardır. Checkpoint otomatik kaydedilir;  
> session bitince `CONTINUE_TRAINING = True` yapıp yeniden başlatın.

---
## 0. Ortam Tespiti ve GPU Kontrolü

In [1]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f"Ortam : {env_name}")

import torch
if not torch.cuda.is_available():
    if IS_LOCAL and torch.backends.mps.is_available():
        print("GPU   : Apple Silicon MPS (nnUNet train MPS'i desteklemez; preprocess/NIfTI çalışır)")
    else:
        raise RuntimeError(
            "GPU bulunamadı!\n"
            "Kaggle: Settings → Accelerator → GPU\n"
            "Colab : Runtime → Change runtime type → GPU"
        )
else:
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"PyTorch CUDA: {torch.version.cuda}")

Ortam : Kaggle
GPU   : Tesla P100-PCIE-16GB
VRAM  : 17.1 GB
PyTorch CUDA: 12.8


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


---
## 1. Ortam Kurulumu

- **Kaggle**: Sadece kütüphane kurulumu
- **Colab**: Kaggle API kimlik doğrulama + Google Drive bağlantısı

In [2]:
if IS_COLAB:
    # ── Kaggle API ─────────────────────────────────────────────────────────
    # Colab sidebar'dan 🔑 simgesine tıklayın ve KAGGLE_USERNAME, KAGGLE_KEY ekleyin
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print("Kaggle kimlik bilgileri Colab Secrets'tan yüklendi")
    except Exception:
        from google.colab import files
        import json as _j, shutil as _sh
        print("kaggle.json dosyasını seçin:")
        _up = files.upload()
        _kp = Path.home() / '.kaggle' / 'kaggle.json'
        _kp.parent.mkdir(parents=True, exist_ok=True)
        for fname in _up:
            _sh.move(fname, str(_kp))
        os.chmod(str(_kp), 0o600)
        _kd = _j.loads(_kp.read_text())
        os.environ['KAGGLE_USERNAME'] = _kd['username']
        os.environ['KAGGLE_KEY']      = _kd['key']
        print("kaggle.json yüklendi")

    # ── Google Drive ────────────────────────────────────────────────────────
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive bağlandı")

else:
    print("Kaggle ortamı — API kurulumu ve Drive atlandı")

Kaggle ortamı — API kurulumu ve Drive atlandı


---
## 2. Kütüphane Kurulumu

In [3]:
print("Kütüphaneler kuruluyor...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'nnunetv2', 'SimpleITK', 'pydicom', 'scipy', 'tqdm'],
    check=True
)
import importlib; importlib.invalidate_caches()

import nnunetv2, SimpleITK, pydicom, scipy, tqdm

import shutil, sysconfig
def find_nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [sysconfig.get_path('scripts'), str(Path(sys.executable).parent),
              '/usr/local/bin', '/opt/conda/bin']:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name

for cmd in ['nnUNetv2_plan_and_preprocess', 'nnUNetv2_train', 'nnUNetv2_predict']:
    p = find_nnunet_cmd(cmd)
    print(f"  {cmd}: {Path(p).exists()}")

Kütüphaneler kuruluyor...
  nnUNetv2_plan_and_preprocess: True
  nnUNetv2_train: True
  nnUNetv2_predict: True


---
## 3. Konfigürasyon

**Kaggle**: Dataset slug'ını `KAGGLE_DATASET_SLUG` ile belirtin (input dizinindeki klasör adı).  
**Colab**: Dataset Kaggle'dan indirilir, Drive'a preprocessed kaydedilir.

In [4]:
IS_KAGGLE

True

In [5]:
import os, sys, json, shutil, time, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List

# ─── Kullanıcı Ayarları ────────────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen'
FOLD         = 0
DATASET_ID   = 100
DATASET_NAME = 'Abdomen'
GITHUB_URL   = 'https://github.com/ramazan2020/abdomen1.git'
# ──────────────────────────────────────────────────────────────────────────


In [ ]:

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

if IS_KAGGLE:
    DATA_DIR    = Path('/kaggle/input/datasets/ramazan2020') / KAGGLE_DATASET_SLUG
    WORK_DIR    = Path('/kaggle/working')
    TMP_DIR     = Path('/kaggle/tmp')   # 60 GB — geçici dosyalar buraya
    # Geçici, her session'da yeniden üretilen büyük dosyalar → /kaggle/tmp
    NIFTI_DIR             = TMP_DIR  / 'nifti'
    NNUNET_RAW            = TMP_DIR  / 'nnunet_raw'
    NNUNET_PREPROCESSED_P = TMP_DIR  / 'nnunet_preprocessed'
    # Model checkpoint'leri working'de kalsın (kayıtlı çıktı, 20 GB)
    NNUNET_RESULTS_P      = WORK_DIR / 'nnunet_results'
    _checkpoint_dataset   = Path('/kaggle/input/nnunet-checkpoint')
    CHECKPOINT_INPUT = _checkpoint_dataset if _checkpoint_dataset.exists() else None

elif IS_COLAB:
    DATA_DIR    = Path('/content/kaggle_data')
    DRIVE_BASE  = Path('/content/drive/MyDrive/Abdomen')
    NIFTI_DIR   = Path('/content/nifti')
    NNUNET_RAW            = DRIVE_BASE / 'nnunet_raw'
    NNUNET_PREPROCESSED_D = DRIVE_BASE / 'nnunet_preprocessed'
    NNUNET_PREPROCESSED_L = Path('/content/nnunet_preprocessed')
    NNUNET_RESULTS_D      = DRIVE_BASE / 'nnunet_results'
    NNUNET_RESULTS_L      = Path('/content/nnunet_results')
    NNUNET_PREPROCESSED_P = NNUNET_PREPROCESSED_L
    NNUNET_RESULTS_P      = NNUNET_RESULTS_L
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    PROJECT_ROOT = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
    DATA_DIR    = PROJECT_ROOT   # splits, manifest burada
    WORK_DIR    = PROJECT_ROOT / 'outputs'
    NIFTI_DIR   = WORK_DIR / 'nifti'
    NNUNET_RAW            = WORK_DIR / 'nnunet_raw'
    NNUNET_PREPROCESSED_P = WORK_DIR / 'nnunet_preprocessed'
    NNUNET_RESULTS_P      = WORK_DIR / 'nnunet_results'

EGITIM_DIR  = Path(os.environ.get('ABDOMEN_TRAIN_DIR', str(DATA_DIR / 'Egitim Verisi')))
YARISMA_DIR = Path(os.environ.get('ABDOMEN_TEST_DIR',  str(DATA_DIR / 'Test Verisi')))
BILGI_XLSX    = Path(os.environ.get("ABDOMEN_BILGI_XLSX", str(DATA_DIR / "Bilgi.xlsx")))

SPLIT_DIR   = WORK_DIR / 'outputs' / 'splits'
if IS_LOCAL:
    SPLIT_DIR = WORK_DIR / 'splits'

DATASET_DIR = NNUNET_RAW / f'Dataset{DATASET_ID}_{DATASET_NAME}'

for p in [
    NIFTI_DIR, NNUNET_RAW, NNUNET_PREPROCESSED_P, NNUNET_RESULTS_P,
    DATASET_DIR / 'imagesTr', DATASET_DIR / 'labelsTr',
    SPLIT_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw']          = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED_P)
os.environ['nnUNet_results']      = str(NNUNET_RESULTS_P)
os.environ['OMP_NUM_THREADS']     = '1'
os.environ['ABDOMEN_SPLIT_DIR']   = str(SPLIT_DIR)
os.environ['ABDOMEN_TRAIN_DIR']   = str(EGITIM_DIR)
os.environ['ABDOMEN_TEST_DIR']    = str(YARISMA_DIR)
os.environ['ABDOMEN_OUT_DIR']     = str(WORK_DIR / 'abdomen_src_out')
os.environ['ABDOMEN_BILGI_XLSX']     = str(BILGI_XLSX)


print(f'Ortam      : {env_name}')
print(f'DATA_DIR   : {DATA_DIR}  (var={DATA_DIR.exists()})')
print(f'EGITIM_DIR : {EGITIM_DIR}  (var={EGITIM_DIR.exists()})')
print(f'SPLIT_DIR  : {SPLIT_DIR}  (var={SPLIT_DIR.exists()})')
if IS_KAGGLE:
    print(f'TMP_DIR    : {TMP_DIR}  (NIfTI + nnUNet raw/preprocessed)')

if IS_KAGGLE and not EGITIM_DIR.exists():
    raise FileNotFoundError(
        f'Dataset bulunamadı: {DATA_DIR}\n'
        f'Kaggle sağ panelden Datasets → "{KAGGLE_DATASET_SLUG}" ekleyin.'
    )


In [7]:
BILGI_XLSX

PosixPath('/kaggle/input/datasets/ramazan2020/abdomen/Bilgi.xlsx')

---
## 4. Veri İndirme (Yalnızca Colab)

**Kaggle'da bu hücre otomatik atlanır.**

In [8]:
if IS_KAGGLE:
    print("Kaggle: Dataset zaten mount edilmiş, indirme atlandı")
    print(f"  Egitim Verisi: {len(list(EGITIM_DIR.iterdir()))} vaka")
else:
    # Colab: Kaggle'dan indir
    _already = EGITIM_DIR.exists() and any(EGITIM_DIR.iterdir())
    if _already:
        print(f"Veri zaten mevcut: {len(list(EGITIM_DIR.iterdir()))} vaka")
    else:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Dataset indiriliyor: {KAGGLE_DATASET_ID}")
        print("Bu işlem 30-90 dakika sürebilir...")
        t0 = time.time()
        r = subprocess.run(
            ['kaggle', 'datasets', 'download',
             '-d', KAGGLE_DATASET_ID,
             '-p', str(DATA_DIR), '--unzip'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('HATA:', r.stderr[-1000:])
            raise RuntimeError('İndirme başarısız')
        print(f"İndirildi ({time.time()-t0:.0f}s)")

    # Colab: preprocessed Drive'da varsa lokal kopyasını al
    _drive_pre = NNUNET_PREPROCESSED_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _local_pre = NNUNET_PREPROCESSED_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _drive_pre.exists() and not _local_pre.exists():
        print('Preprocessed Drive→Local kopyalanıyor...')
        t0 = time.time()
        shutil.copytree(str(_drive_pre), str(_local_pre))
        print(f'Kopyalandı ({time.time()-t0:.0f}s)')
        os.environ['nnUNet_results'] = str(NNUNET_RESULTS_L)

assert EGITIM_DIR.exists(), f'EGITIM_DIR bulunamadı: {EGITIM_DIR}'
assert SPLIT_DIR.exists(),  f'SPLIT_DIR bulunamadı: {SPLIT_DIR}'
print("Veri doğrulandı")

Kaggle: Dataset zaten mount edilmiş, indirme atlandı
  Egitim Verisi: 736 vaka
Veri doğrulandı


In [9]:
# ── GitHub Repo / Local src ───────────────────────────────────────────────
if IS_LOCAL:
    # Local: src/ proje kökünde zaten var, klonlamaya gerek yok
    PROJECT_ROOT = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
    REPO_DIR = PROJECT_ROOT
    print(f'Local: src/ kullanılıyor → {REPO_DIR / "src"}')
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        print(f'Klonlanıyor: {GITHUB_URL}')
        subprocess.run(
            ['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)],
            check=True
        )
    else:
        print('Repo mevcut, güncelleniyor...')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       capture_output=True)
    print(f'Repo : {REPO_DIR}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.splits     import load_fold, raw_case_id
from src.lifting    import BboxLifter
from src.evaluation import f1_at_iou, top5_f1_mean

print('src.splits      ✓')
print('src.lifting     ✓')
print('src.evaluation  ✓')

Repo mevcut, güncelleniyor...
Repo : /kaggle/working/abdomen1
src.splits      ✓
src.lifting     ✓
src.evaluation  ✓


In [10]:
SPLIT_DIR

PosixPath('/kaggle/working/outputs/splits')

In [11]:
# ── Veri Hazırlık Kontrolü ─────────────────────────────────────────────────
# manifest.csv + splits, Faz1_2_VeriHazirlik notebook'u tarafından üretilir.
# Mevcut değilse burada otomatik oluşturulur.

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
_manifest_csv = SPLIT_DIR / 'manifest.csv'
_splits_csv   = SPLIT_DIR / 'splits.csv'

if not _manifest_csv.exists():
    print('manifest.csv bulunamadı — oluşturuluyor...')
    _bilgi = DATA_DIR / 'Bilgi.xlsx'
    if not _bilgi.exists():
        raise FileNotFoundError(
            f'Bilgi.xlsx bulunamadı: {_bilgi}\n'
            "Faz1_2_VeriHazirlik notebook'unu önce çalıştırın."
        )
    os.environ.setdefault('ABDOMEN_BILGI_XLSX', str(_bilgi))
    from src.preprocessing import build_manifest
    build_manifest(_manifest_csv)
    print(f'manifest.csv oluşturuldu ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')
else:
    print(f'manifest.csv mevcut ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')

if not _splits_csv.exists():
    print('splits.csv bulunamadı — oluşturuluyor...')
    from src.splits import make_splits
    make_splits(out_dir=SPLIT_DIR)
    print('splits.csv oluşturuldu ✓')
else:
    print('splits.csv mevcut ✓')

_fold_csv = SPLIT_DIR / f'fold{FOLD}_train.csv'
if not _fold_csv.exists():
    print(f'fold{FOLD}_train.csv eksik — make_splits yeniden çalıştırılıyor...')
    from src.splits import make_splits
    make_splits(out_dir=SPLIT_DIR)
print(f'fold{FOLD}_train.csv: {"✓ mevcut" if _fold_csv.exists() else "⚠ hâlâ eksik"}')

manifest.csv bulunamadı — oluşturuluyor...
Manifest yazıldı: /kaggle/working/outputs/splits/manifest.csv  (satır: 39268)
  BBox annotasyonu olan kesit  : 31,979  (YOLO girdisi)
  Boundary Slice olan kesit    : 7,898  (segmentasyon girdisi)
  Toplam BBox annotasyonu      : 34,550
  Toplam Boundary Slice annot. : 7,898
manifest.csv oluşturuldu ✓  (4730 KB)
splits.csv bulunamadı — oluşturuluyor...
splits.csv oluşturuldu ✓
fold0_train.csv: ✓ mevcut


---
## 5. Yardımcı Fonksiyonlar

In [12]:
manifest = pd.read_csv(SPLIT_DIR / 'manifest.csv')

# BboxLifter: 2D annotasyonları 3D'ye yükseltir (src.lifting)
_lifter = BboxLifter(
    manifest,
    egitim_dir=EGITIM_DIR,
    yarisma_dir=YARISMA_DIR,
)

def lift_2d_to_3d(manifest_df, case, delta_z=2, iou_th=0.3):
    """src.lifting.BboxLifter için geriye uyumlu sarmalayıcı."""
    _l = BboxLifter(manifest_df, egitim_dir=EGITIM_DIR,
                    yarisma_dir=YARISMA_DIR, delta_z=delta_z, iou_th=iou_th)
    return _l.lift(case)

train_cases = load_fold(FOLD, 'train')
val_cases   = load_fold(FOLD, 'val')
all_cases   = sorted(set(train_cases + val_cases))

print(f'Manifest  : {len(manifest):,} satır')
print(f'Fold {FOLD} : {len(train_cases)} train + {len(val_cases)} val = {len(all_cases)} toplam')
print('src.lifting.BboxLifter hazır ✓')

Manifest  : 39,268 satır
Fold 0 : 580 train + 145 val = 725 toplam
src.lifting.BboxLifter hazır ✓


In [13]:
# f1_at_iou ve top5_f1_mean src.evaluation'dan import edildi (cell_github)
# İmzalar:
#   f1_at_iou(pred_df, gt_df, iou_th)  → {'per_class', 'macro_f1', 'micro_f1'}
#   top5_f1_mean(pred_df, gt_df)        → {'per_threshold', 'top5', 'top5_mean_f1'}
print('src.evaluation.f1_at_iou     ✓')
print('src.evaluation.top5_f1_mean  ✓')

src.evaluation.f1_at_iou     ✓
src.evaluation.top5_f1_mean  ✓


---
## 6. DICOM → NIfTI Dönüşümü

**Kaggle**: Her session'da yeniden çalışır (working disk session ömürlüdür).  
Paralel dönüşüm — mevcut dosyalar atlanır.

In [ ]:
import SimpleITK as sitk
import pydicom
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

def dicom_to_nifti(case: str) -> str:
    raw = raw_case_id(case)
    out = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
    if out.exists(): return 'skip'
    case_dir = EGITIM_DIR / str(raw)
    if not case_dir.exists(): return f'missing:{case}'
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(str(case_dir))
    if not names: return f'no_dcm:{case}'

    # Karışık boyutlu serileri filtrele: baskın (Rows×Cols) grubunu al
    # 608×512 gibi anormal boyutlular genellikle bozuk/truncated DICOM'dur;
    # header boyutu söyler ama pixel verisi farklı boyuttadır → atılmaları güvenlidir
    size_map = {}
    for n in names:
        try:
            h = pydicom.dcmread(n, stop_before_pixels=True)
            size_map.setdefault((int(h.Rows), int(h.Columns)), []).append(n)
        except Exception:
            pass
    if len(size_map) > 1:
        dominant = max(size_map.values(), key=len)
        dropped = sum(len(v) for k, v in size_map.items() if v is not dominant)
        names = dominant
        # Atılan dilim sayısını döndür (eğitim için tanı amaçlı)
        return_suffix = f'|dropped:{dropped}'
    else:
        return_suffix = ''

    reader.SetFileNames(names)
    try:
        sitk.WriteImage(reader.Execute(), str(out))
        return 'ok' + return_suffix
    except Exception as e:
        return f'err:{e}'

_n_workers = min(os.cpu_count() or 4, 8)

print(f'DICOM→NIfTI: {len(all_cases)} vaka  |  Kaynak: {EGITIM_DIR}')
print(f'Workers: {_n_workers}  (CPU: {os.cpu_count()})')
_nii_existing = len(list(NIFTI_DIR.glob('*.nii.gz')))
print(f'Mevcut NIfTI: {_nii_existing}')

sitk.ProcessObject.GlobalWarningDisplayOff()
with ThreadPoolExecutor(max_workers=_n_workers) as ex:
    results = list(tqdm(ex.map(dicom_to_nifti, all_cases),
                        total=len(all_cases), desc='DICOM→NIfTI'))
sitk.ProcessObject.GlobalWarningDisplayOn()

ok      = sum(1 for r in results if r.startswith('ok') and 'dropped' not in r)
ok_filt = [r for r in results if r.startswith('ok|dropped:')]
skip    = sum(1 for r in results if r == 'skip')
errs    = [r for r in results if r.startswith('err:')]

print(f'{ok} yeni  |  {len(ok_filt)} boyut filtreli  |  {skip} atlandı  |  {len(errs)} hata')
if ok_filt:
    for r in ok_filt:
        dropped_n = int(r.split('dropped:')[1])
        # hangi vaka olduğunu bulmak için indekse bak
        idx = results.index(r)
        print(f'  → {all_cases[idx]}: {dropped_n} bozuk dilim atıldı (header/pixel boyut uyumsuzluğu)')
if errs:
    print('Hatalar:', errs[:5])
print(f'Toplam NIfTI: {len(list(NIFTI_DIR.glob("*.nii.gz")))}')


---
## 7. nnUNet Dataset Formatı Hazırlama

- `imagesTr/` → CT hacimler (symlink)
- `labelsTr/` → Semantic maske (0=arka plan, 1-6=patoloji sınıfları)
- `dataset.json` → nnUNet meta

In [ ]:
manifest = pd.read_csv(SPLIT_DIR / 'manifest.csv')
print(f'Manifest: {len(manifest)} satır  |  Sütunlar: {list(manifest.columns)}')


def prep_case_nnunet(case: str) -> str:
    raw = raw_case_id(case)
    src_nii = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
    if not src_nii.exists(): return f'missing:{case}'

    dst_img = DATASET_DIR / 'imagesTr' / f'case_{raw:05d}_0000.nii.gz'
    if not dst_img.exists():
        try: os.symlink(src_nii, dst_img)
        except (OSError, NotImplementedError): shutil.copy2(str(src_nii), str(dst_img))

    dst_lbl = DATASET_DIR / 'labelsTr' / f'case_{raw:05d}.nii.gz'
    if dst_lbl.exists(): return 'skip'

    boxes = lift_2d_to_3d(manifest, case)
    img_itk = sitk.ReadImage(str(src_nii))
    shape = sitk.GetArrayFromImage(img_itk).shape  # (Z, Y, X)
    mask = np.zeros(shape, dtype=np.uint8)
    for b in boxes:
        label = int(b['class']) + 1
        mask[int(b['z1']):min(int(b['z2'])+1,shape[0]),
             int(b['y1']):min(int(b['y2'])+1,shape[1]),
             int(b['x1']):min(int(b['x2'])+1,shape[2])] = label
    m = sitk.GetImageFromArray(mask)
    m.CopyInformation(img_itk)
    sitk.WriteImage(m, str(dst_lbl))
    return 'ok'

print(f'Dataset hazırlanıyor: {len(all_cases)} vaka...')
results = [prep_case_nnunet(c) for c in tqdm(all_cases, desc='Dataset prep')]

ok   = results.count('ok')
skip = results.count('skip')
miss = [r for r in results if 'missing' in r]
print(f'{ok} yeni  |  {skip} atlandı  |  {len(miss)} eksik NIfTI')
if miss:
    print('  Eksik NIfTI (dönüşüm başarısız olmuş vakalar):', miss[:5])

n_img = len(list((DATASET_DIR/'imagesTr').glob('*.nii.gz')))
n_lbl = len(list((DATASET_DIR/'labelsTr').glob('*.nii.gz')))
print(f'imagesTr: {n_img}  |  labelsTr: {n_lbl}')
if n_img == 0:
    raise RuntimeError('imagesTr boş!')

# numTraining = gerçek dosya sayısı; len(all_cases) ile fark NIfTI dönüşüm hatası demektir
dataset_json = {
    'channel_names': {'0': 'CT'},
    'labels': {'background': 0, **{n: i+1 for i, n in enumerate(SUPER_CLASSES)}},
    'numTraining': n_img,
    'file_ending': '.nii.gz',
}
(DATASET_DIR / 'dataset.json').write_text(json.dumps(dataset_json, indent=2))
if n_img < len(all_cases):
    print(f'⚠ numTraining={n_img} (beklenen {len(all_cases)}; {len(all_cases)-n_img} NIfTI dönüşümü başarısız)')
else:
    print(f'numTraining={n_img} ✓')


---
## 8. nnUNet Plan + Preprocess

Fingerprint çıkarımı, plan oluşturma ve veriyi nnUNet formatına dönüştürme.  
**Önceki session'da tamamlandıysa bu hücre atlanır.**

In [ ]:
_plans_file = (NNUNET_PREPROCESSED_P
               / f'Dataset{DATASET_ID}_{DATASET_NAME}'
               / 'nnUNetPlans.json')

if _plans_file.exists():
    print(f'Plans dosyası mevcut, preprocess atlandı: {_plans_file}')
else:
    _cmd = find_nnunet_cmd('nnUNetv2_plan_and_preprocess')

    # Kaggle RAM ~13 GB; varsayılan worker sayısı (cpu_count) CT'leri paralel
    # yükleyince OOM olur → fingerprint ve preprocess worker sayısını sınırla
    _np = 2 if IS_KAGGLE else max(1, (os.cpu_count() or 4) // 2)

    print(f'Plan + Preprocess başlatılıyor...')
    print(f'  Dataset: {DATASET_ID}, Config: 3d_fullres, Workers: {_np}')
    print('  ~15–30 dakika sürebilir...')

    _proc = subprocess.Popen(
        [_cmd, '-d', str(DATASET_ID), '-c', '3d_fullres',
         '-npfp', str(_np),    # fingerprint extraction worker sayısı (tek tire)
         '-np', str(_np),      # preprocessing worker sayısı
         '--verify_dataset_integrity'],
        env={**os.environ},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    for line in _proc.stdout:
        print(line, end='', flush=True)
    _proc.wait()
    if _proc.returncode != 0:
        raise RuntimeError(f'plan_and_preprocess başarısız! Kod: {_proc.returncode}')
    print('\nPlan + Preprocess tamamlandı!')

    # Colab: preprocessed'i Drive'a yedekle
    if IS_COLAB:
        _src = NNUNET_PREPROCESSED_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
        _dst = NNUNET_PREPROCESSED_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
        if _src.exists() and not _dst.exists():
            print("Preprocessed Drive'a yedekleniyor...")
            shutil.copytree(str(_src), str(_dst))
            print('Yedeklendi')

# Kaggle: nnunet_raw preprocessing bittikten sonra gereksiz → /kaggle/tmp temizle
if IS_KAGGLE and NNUNET_RAW.exists():
    print(f'\nTemizleniyor (tmp): {NNUNET_RAW}')
    shutil.rmtree(str(NNUNET_RAW))
    print('nnunet_raw silindi ✓')


In [ ]:
# Özel fold split yaz (nnUNet auto-split'ini geçersiz kıl)
_splits_dir = NNUNET_PREPROCESSED_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
_splits_dir.mkdir(parents=True, exist_ok=True)

_splits = [{
    'train': [f'case_{raw_case_id(c):05d}' for c in train_cases],
    'val'  : [f'case_{raw_case_id(c):05d}' for c in val_cases],
}]
(_splits_dir / 'splits_final.json').write_text(json.dumps(_splits, indent=2))
print(f'splits_final.json: {len(train_cases)} train, {len(val_cases)} val')

---
## 9. Eğitim

**Kaggle 9 saat session sınırı**: Eğitim otomatik checkpoint kaydeder.  
Session bittiğinde:
1. `/kaggle/working/nnunet_results/` klasörünü Kaggle output olarak kaydedin
2. Yeni session'da bu output'u dataset olarak ekleyin
3. `CONTINUE_TRAINING = True` yapıp çalıştırın

In [ ]:
CONTINUE_TRAINING = False  # Önceki session'dan devam etmek için True yapın

if IS_KAGGLE and CONTINUE_TRAINING:
    # Önceki session checkpoint'ini working dizinine kopyala
    _ckpt_src = Path('/kaggle/input/nnunet-checkpoint')
    if _ckpt_src.exists():
        print(f'Checkpoint yükleniyor: {_ckpt_src}')
        _ckpt_dst = NNUNET_RESULTS_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
        if not _ckpt_dst.exists():
            t0 = time.time()
            shutil.copytree(str(_ckpt_src / f'Dataset{DATASET_ID}_{DATASET_NAME}'),
                            str(_ckpt_dst))
            print(f'Checkpoint kopyalandı ({time.time()-t0:.0f}s)')
        else:
            print('Checkpoint zaten mevcut')
    else:
        print(f'Checkpoint dataset bulunamadı: {_ckpt_src}')
        print('Sağ panelden Datasets → nnunet-checkpoint ekleyin')
elif IS_COLAB and CONTINUE_TRAINING:
    _ckpt_drive = NNUNET_RESULTS_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _ckpt_local = NNUNET_RESULTS_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _ckpt_drive.exists() and not _ckpt_local.exists():
        print('Checkpoint Drive\'dan yükleniyor...')
        shutil.copytree(str(_ckpt_drive), str(_ckpt_local))
        print('Yüklendi')

print(f'CONTINUE_TRAINING: {CONTINUE_TRAINING}')

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU gerekli!'

_cmd = find_nnunet_cmd('nnUNetv2_train')
print(f'=== Eğitim Başlıyor ===')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'Dataset : {DATASET_ID}, Config: 3d_fullres, Fold: {FOLD}')
print(f'Devam   : {CONTINUE_TRAINING}')
print(f'Results : {os.environ["nnUNet_results"]}')
print('─' * 50)

_args = [_cmd, str(DATASET_ID), '3d_fullres', str(FOLD), '--npz']
if CONTINUE_TRAINING:
    _args.append('--c')

_proc = subprocess.Popen(
    _args, env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()

print('─' * 50)
if _proc.returncode == 0:
    print('Eğitim tamamlandı!')
else:
    print(f'Eğitim kodu: {_proc.returncode} (session kesildi veya hata)')

# Colab: modeli Drive'a yedekle
if IS_COLAB:
    _src = NNUNET_RESULTS_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _dst = NNUNET_RESULTS_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _src.exists():
        if _dst.exists(): shutil.rmtree(str(_dst))
        shutil.copytree(str(_src), str(_dst))
        print(f"Model Drive'a yedeklendi: {_dst}")

# Kaggle: eğitim başarıyla tamamlandıysa preprocessed artık gerekmez → /kaggle/tmp temizle
# (session kesilirse returncode != 0 → silinmez, bir sonraki session'da --c ile devam edilebilir)
if IS_KAGGLE and _proc.returncode == 0 and NNUNET_PREPROCESSED_P.exists():
    print(f'\nTemizleniyor (tmp): {NNUNET_PREPROCESSED_P}')
    shutil.rmtree(str(NNUNET_PREPROCESSED_P))
    print('nnunet_preprocessed silindi ✓')


---
## 10. Tahmin (Inference) — Validasyon Seti

In [ ]:
VAL_INPUT_DIR  = NNUNET_RESULTS_P.parent / 'val_input'
VAL_OUTPUT_DIR = (NNUNET_RESULTS_P
                  / f'Dataset{DATASET_ID}_{DATASET_NAME}'
                  / 'nnUNetTrainer__nnUNetPlans__3d_fullres'
                  / f'fold_{FOLD}'
                  / 'val_predictions')
VAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
VAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_linked = 0
for c in val_cases:
    raw = raw_case_id(c)
    src = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
    dst = VAL_INPUT_DIR / f'case_{raw:05d}_0000.nii.gz'
    if src.exists() and not dst.exists():
        try: os.symlink(src, dst)
        except (OSError, NotImplementedError): shutil.copy2(str(src), str(dst))
        _linked += 1

print(f'Val input: {len(list(VAL_INPUT_DIR.glob("*.nii.gz")))} dosya')
print(f'Tahmin çıktı: {VAL_OUTPUT_DIR}')

_cmd = find_nnunet_cmd('nnUNetv2_predict')
print('Tahmin başlatılıyor...')

_proc = subprocess.Popen(
    [_cmd,
     '-i', str(VAL_INPUT_DIR),
     '-o', str(VAL_OUTPUT_DIR),
     '-d', str(DATASET_ID),
     '-c', '3d_fullres',
     '-f', str(FOLD),
     '--save_probabilities'],
    env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()

_preds = list(VAL_OUTPUT_DIR.glob('*.nii.gz'))
print(f'\nTahmin tamamlandı: {len(_preds)} NIfTI mask')

# Kaggle: inference tamamlandı → NIfTI ve val_input symlink'leri artık gerekmez → /kaggle/tmp temizle
if IS_KAGGLE and _proc.returncode == 0 and len(_preds) > 0:
    if NIFTI_DIR.exists():
        print(f'\nTemizleniyor (tmp): {NIFTI_DIR}')
        shutil.rmtree(str(NIFTI_DIR))
        print('nifti silindi ✓')
    if VAL_INPUT_DIR.exists():
        shutil.rmtree(str(VAL_INPUT_DIR))
        print('val_input silindi ✓')


---
## 11. Değerlendirme — F1 Skoru

In [ ]:
from scipy import ndimage

def seg_to_pred_df(pred_dir: Path) -> pd.DataFrame:
    """NIfTI segmentasyon maskeleri → prediction DataFrame (2D projeksiyon)."""
    rows = []
    for nii_path in sorted(pred_dir.glob('*.nii.gz')):
        try: raw = int(nii_path.stem.split('_')[1])
        except: continue
        mask = sitk.GetArrayFromImage(sitk.ReadImage(str(nii_path)))  # [Z,Y,X]
        prob_path = nii_path.with_suffix('').with_suffix('.npz')
        probs = np.load(str(prob_path))['probabilities'] if prob_path.exists() else None
        for label_id in range(1, len(SUPER_CLASSES)+1):
            binary = (mask == label_id).astype(np.uint8)
            if binary.sum() == 0: continue
            labeled, n_comp = ndimage.label(binary)
            total_vox = float(binary.sum())
            for comp_id in range(1, n_comp+1):
                cm = (labeled == comp_id)
                coords = np.where(cm)
                z1,z2 = int(coords[0].min()),int(coords[0].max())
                y1,y2 = int(coords[1].min()),int(coords[1].max())
                x1,x2 = int(coords[2].min()),int(coords[2].max())
                score = float(probs[label_id][cm].mean()) if probs is not None else float(cm.sum())/total_vox
                for z in range(z1, z2+1):
                    rows.append({'case':raw,'image_id':z,'class':label_id-1,'score':score,
                                 'x1':float(x1),'y1':float(y1),'x2':float(x2),'y2':float(y2)})
    return pd.DataFrame(rows)

pred_df = seg_to_pred_df(VAL_OUTPUT_DIR)
print(f'Prediction: {len(pred_df):,} satır, {pred_df["case"].nunique() if not pred_df.empty else 0} vaka')

# Ground-truth
gt_rows = []
for case in val_cases:
    raw = raw_case_id(case)
    for b in lift_2d_to_3d(manifest, case):
        for z in range(int(b['z1']), int(b['z2'])+1):
            gt_rows.append({'case':raw,'image_id':z,'class':int(b['class']),
                            'x1':float(b['x1']),'y1':float(b['y1']),
                            'x2':float(b['x2']),'y2':float(b['y2'])})
gt_df = pd.DataFrame(gt_rows)
print(f'GT        : {len(gt_df):,} satır, {gt_df["case"].nunique() if not gt_df.empty else 0} vaka')

In [ ]:
if pred_df.empty:
    print('Prediction boş — inference adımını kontrol edin.')
else:
    top5   = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)

    print(f'=== nnUNet v2 — Fold {FOLD} Sonuçları ===')
    print(f'Top-5 Mean F1 (makale metriği) : {top5["top5_mean_f1"]:.4f}')
    print(f'Macro F1 @ IoU=0.3             : {detail["macro_f1"]:.4f}')
    print(f'Micro F1 @ IoU=0.3             : {detail["micro_f1"]:.4f}')
    print()
    print('IoU eşiğine göre Macro F1:')
    for th, f1v in top5['per_threshold']:
        print(f'  {th:.1f}: {f1v:.4f}')
    print()
    print('Sınıf bazında @ IoU=0.3:')
    for cls_id, cls_name in enumerate(SUPER_CLASSES):
        if cls_id in detail['per_class']:
            m = detail['per_class'][cls_id]
            print(f'  {cls_id} {cls_name:<30}  '
                  f'P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}  '
                  f'TP={m["tp"]} FP={m["fp"]} FN={m["fn"]}')

---
## 12. Sonuçları Kaydet

**Kaggle**: Çıktılar `/kaggle/working/` altında otomatik kaydedilir.  
Notebook'u commit edin → "Save Version" → output olarak kullanılabilir hale gelir.  
**Colab**: Drive'a kopyalanır.

In [ ]:
OUTPUT_DIR = NNUNET_RESULTS_P.parent / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV'ler
if not pred_df.empty:
    pred_df.to_csv(OUTPUT_DIR / f'pred_fold{FOLD}.csv', index=False)
    gt_df.to_csv(OUTPUT_DIR   / f'gt_fold{FOLD}.csv',   index=False)

    summary = {
        'fold': FOLD,
        'val_cases': len(val_cases),
        'top5_mean_f1': top5['top5_mean_f1'],
        'macro_f1_03' : detail['macro_f1'],
        'micro_f1_03' : detail['micro_f1'],
        'per_threshold': {str(th): float(f1v) for th, f1v in top5['per_threshold']},
    }
    (OUTPUT_DIR / f'summary_fold{FOLD}.json').write_text(json.dumps(summary, indent=2))
    print(f'Çıktılar: {OUTPUT_DIR}')
    for f in sorted(OUTPUT_DIR.glob('*')):
        print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

# Colab: Drive'a kopyala
if IS_COLAB:
    _dst = DRIVE_BASE / 'output'
    if _dst.exists(): shutil.rmtree(str(_dst))
    shutil.copytree(str(OUTPUT_DIR), str(_dst))
    print(f'\nDrive\'a kopyalandı: {_dst}')

print('\nTamamlandı!')
if IS_KAGGLE:
    print('\nCheckpoint kaydetmek için:')
    print('  Save Version → sağ üst köşe → Save & Run All')
    print('  Çıktılar otomatik /kaggle/working/ altında saklanır')